In [0]:
df = spark.read.parquet('/Volumes/data/orders/files/result/eco.parquet/')
df2 = df.select('user_id').count()
display(df2)

In [0]:
# Count users by gender. 
import pyspark.sql.functions as f
df = spark.read.parquet('/Volumes/data/orders/files/result/eco.parquet/')
df = df.groupBy('gender').agg(f.count('user_id').alias('count'))
display(df)

In [0]:
# Average time_on_site per device_type. 
import pyspark.sql.functions as f
df = spark.read.parquet('/Volumes/data/orders/files/result/eco.parquet/')
df = df.groupBy('device_type').agg(f.avg('time_on_site').alias('avg_time'))
display(df)

In [0]:
# Total purchases by device_type. 
import pyspark.sql.functions as f
df = spark.read.parquet('/Volumes/data/orders/files/result/eco.parquet/')
df = df.groupBy('device_type').agg(f.sum('purchase'))
display(df)

In [0]:
 # Count users who saw discount vs not. 
import pyspark.sql.functions as f
df = spark.read.parquet('/Volumes/data/orders/files/result/eco.parquet/')
df = df.groupBy('discount_seen').agg(f.count('user_id'))
display(df)

In [0]:
'''Create a column: 
"High Engagement" if pages_viewed > 5 
else "Low Engagement" 
'''
import pyspark.sql.functions as f
df = spark.read.parquet('/Volumes/data/orders/files/result/eco.parquet/')
df = df.withColumn('engagement', f.when(df.pages_viewed > 5, 'High Engagement').otherwise('Low Engagement'))\
    .select('user_id','engagement')
display(df)

In [0]:
'''Categorize users: 
"Buyer" if purchase = 1 
else "Non-Buyer" 
'''
import pyspark.sql.functions as f
df = spark.read.parquet('/Volumes/data/orders/files/result/eco.parquet/')
df = df.withColumn('user_type',f.when(df.purchase == 1,'Buyer').otherwise('Non-Buyer'))\
    .select('user_id','user_type')
display(df)

In [0]:
'''Create discount effectiveness: 
"Effective" if discount_seen = 1 AND purchase = 1 
'''
import pyspark.sql.functions as f
df = spark.read.parquet('/Volumes/data/orders/files/result/eco.parquet/')
df=df.withColumn('effect',f.when((f.col('discount_seen')==1)&(f.col('purchase')==1),'Effective')\
    .otherwise('non-effective')).select('user_id','effect')
display(df)

In [0]:
# Avg time_on_site for buyers vs non-buyers 
import pyspark.sql.functions as f
df = spark.read.parquet('/Volumes/data/orders/files/result/eco.parquet/')
df = df.withColumn('customer_type',f.when(df.purchase == 1,'Buyer').otherwise('Non-Buyer'))
avg = df.groupBy('customer_type').agg(f.avg('time_on_site').alias('avg_time'))
display(avg)

In [0]:
# Bounce rate avg by device_type 
import pyspark.sql.functions as f
df = spark.read.parquet('/Volumes/data/orders/files/result/eco.parquet/')
avg = df.groupBy('device_type').agg(f.avg('bounce_rate').alias('avg_bounce'))
display(avg)

In [0]:
'''Calculate conversion rate: 
total purchases / total users
Conversion rate by device_type 
'''
import pyspark.sql.functions as f
df = spark.read.parquet('/Volumes/data/orders/files/result/eco.parquet/')
df = df.groupBy('device_type').agg(f.sum('purchase').alias('total_purchases'),\
    f.count('user_id').alias('total_users'))
df = df.withColumn('conversion_rate',f.col('total_purchases')/f.col('total_users'))\
    .select("device_type","conversion_rate")
display(df)

In [0]:
# Conversion rate for returning vs new users 
import pyspark.sql.functions as f
df = spark.read.parquet('/Volumes/data/orders/files/result/eco.parquet/')
df = df.withColumn('user_type', f.when(df.returning_user == 1, 'old_user').otherwise('new_user'))
df = df.groupBy('user_type').agg(f.sum('purchase').alias('total_purchases'),\
     f.count('user_id').alias('total_users'))
df = df.withColumn('conversion_rate', f.col('total_purchases') / f.col('total_users'))\
    .select("user_type", "conversion_rate")
display(df)

In [0]:
'''Find top 5 users with highest time_on_site '''
import pyspark.sql.functions as f
from pyspark.sql.window import Window
df = spark.read.parquet('/Volumes/data/orders/files/result/eco.parquet/')
w = Window.orderBy(f.col('total_time').desc())
df = df.groupBy('user_id').agg(f.sum(f.col('time_on_site')).alias('total_time'))\
    .withColumn('rnk', f.rank().over(w))\
    .filter(f.col('rnk') <= 5)\
    .select('user_id','total_time','rnk')
display(df)

In [0]:
'''Find users with high engagement but no purchase
(pages_viewed > 5 AND purchase = 0) '''
import pyspark.sql.functions as f
df = spark.read.parquet('/Volumes/data/orders/files/result/eco.parquet/')
df = df.filter((f.col('pages_viewed') > 5) & (f.col('purchase') == 0))\
    .select('user_id')
display(df)

In [0]:
'''Find correlation-like insight: 
Avg pages_viewed for buyers vs non-buyers 
'''
import pyspark.sql.functions as f
df = spark.read.parquet('/Volumes/data/orders/files/result/eco.parquet/')
df = df.withColumn('user_type',f.when(df.purchase == 1,'Buyer').otherwise('Non-Buyer'))
df = df.groupBy('user_type').agg(f.avg('pages_viewed').alias('avg_pages'))
display(df)

In [0]:
'''Find users who: 
clicked ad 
saw discount 
but didnt purchase 
'''
import pyspark.sql.functions as f
df = spark.read.parquet('/Volumes/data/orders/files/result/eco.parquet/')
df = df.filter((f.col('ad_clicked') == 1) & (f.col('discount_seen') == 1) & (f.col('purchase') == 0))\
    .select('user_id')
display(df)

In [0]:
'''Find % of users who: 
added items to cart but didnt purchase '''
import pyspark.sql.functions as f
df = spark.read.parquet('/Volumes/data/orders/files/result/eco.parquet/')
df = df.select((f.sum(f.when((f.col('cart_items') > 0) & (f.col('purchase') == 0), 1)\
    .otherwise(0))/f.count('user_id')).alias('per'))
display(df)

In [0]:
# Rank users by time_on_site 
import pyspark.sql.functions as f
from pyspark.sql.window import Window
df = spark.read.parquet('/Volumes/data/orders/files/result/eco.parquet/')
w = Window.orderBy(f.col('time_on_site').desc())
df = df.withColumn('rnk', f.rank().over(w))\
    .select('user_id','time_on_site','rnk')
display(df)

In [0]:
# Running average of pages_viewed by user_id 
import pyspark.sql.functions as f
from pyspark.sql.window import Window
df = spark.read.parquet('/Volumes/data/orders/files/result/eco.parquet/')
w = Window.partitionBy('user_id').orderBy('avg_session_time')\
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
df = df.withColumn('running_avg_pages', f.avg('pages_viewed').over(w))
display(df)

In [0]:
# Find top user per device_type 
import pyspark.sql.functions as f
from pyspark.sql.window import Window
df = spark.read.parquet('/Volumes/data/orders/files/result/eco.parquet/')
w = Window.partitionBy('device_type').orderBy(f.col('time_on_site').desc())
df = df.withColumn('rnk', f.rank().over(w)).select('user_id','device_type','time_on_site')\
    .filter(f.col('rnk') == 1)
display(df)

In [0]:
# Dense rank users by avg_session_time 
import pyspark.sql.functions as f
from pyspark.sql.window import Window
df = spark.read.parquet('/Volumes/data/orders/files/result/eco.parquet/')
w = Window.orderBy(f.col('avg_session_time').desc())
df = df.withColumn('dence_rnk', f.dense_rank().over(w)).select('user_id','avg_session_time','dence_rnk')
display(df)